<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Criterios de calidad de los productos priorizados

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `funciones.nearest_product`: identificación del producto más cercano en el espacio de producto.

In [1]:
import pandas as pd
import numpy as np
from funciones.nearest_product import nearest_product

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables

### Parámetros

Define el decil a partir del cual se considera que un producto alcanza calidad alta, el año de
referencia y el umbral de proximidad para buscar productos comparables.

In [3]:
decil_alto = 8

anio_referencia = 2023

cercania = 0.65

### Lectura de datos

Carga la cartera con tamaño de mercado, el comercio bilateral, la matriz de proximidad y las
dos estimaciones de calidad: la del período completo y la del año base de comparación.

In [4]:
df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')
df_comercio = pd.read_parquet(temp / 'df_comercio.parquet')
df_product_space = pd.read_parquet(temp / 'df_product_space.parquet')
df_calidad = pd.read_parquet(temp / 'df_calidad.parquet')
df_calidad_13 = pd.read_parquet(temp / 'df_calidad_13.parquet')
df_hs_names = pd.read_parquet(temp / 'df_hs_names.parquet')

carpeta_calidad = tables / 'calidad'
carpeta_calidad.mkdir(parents=True, exist_ok=True)
(carpeta_calidad / 'otros países').mkdir(parents=True, exist_ok=True)

### Calidad de los productos que Colombia ya exporta

Restringe la estimación de calidad a las exportaciones colombianas del año de referencia y a
los productos clasificados como oportunidades a profundizar.

In [5]:
df_calidad_margen_intensivo = df_calidad.query("`ISO Exportador` == 'COL' and Año == @anio_referencia")[['ISO Exportador','HS6','ISO Importador','Grupo Importador','Decile']].merge(df_products.query("Margen == 'Oportunidades a profundizar'")[['HS6']].drop_duplicates(), how='inner', on='HS6')

### Criterio 1: calidad alta en varios grupos de destino

Cuenta en cuántos grupos geoeconómicos distintos el producto alcanza el decil alto de calidad.

In [6]:
df_output1 = df_products.merge(df_calidad_margen_intensivo.query("Decile >= @decil_alto").groupby('HS6')['Grupo Importador'].nunique().reset_index(),how='inner', on='HS6').drop(columns=['Fuente','Categoría']).rename(columns={'Grupo Importador':'Grupos'}).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante'], ascending=[False, False])

df_output1.to_excel(carpeta_calidad / 'criterio1.xlsx', index=False)

### Criterio 2: calidad alta en un destino y baja en otro

Identifica los productos que alcanzan el decil alto en un mercado y no lo alcanzan en otro, lo
que señala margen de mejora en ese segundo destino.

In [7]:
df_criterio2 = df_calidad_margen_intensivo.merge(df_calidad_margen_intensivo, on=['ISO Exportador','HS6'], how='left', suffixes=('_j', '_k')).query("`ISO Importador_j` != `ISO Importador_k`").drop_duplicates().dropna().query("Decile_j >= @decil_alto and Decile_k < @decil_alto").groupby(['HS6','ISO Importador_j','Grupo Importador_j','ISO Importador_k','Grupo Importador_k'])[['Decile_j','Decile_k']].mean().reset_index()

df_output2 = df_products.merge(df_criterio2.groupby(['HS6']).agg({'Grupo Importador_k':'nunique','Decile_j':'mean'}).round().reset_index(),how='inner', on='HS6').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante'], ascending=[False, False])

df_output2.to_excel(carpeta_calidad / 'criterio2.xlsx', index=False)

### Criterio 3: mejora de calidad frente al año base

Compara el decil del año de referencia con el del año base y conserva los productos que pasaron
de calidad baja a calidad alta.

In [8]:
df_criterio3 = df_calidad_margen_intensivo.merge(df_calidad_13.query("`ISO Exportador` == 'COL'")[['ISO Exportador','HS6','ISO Importador','Grupo Importador','Decile']], on=['ISO Exportador','HS6','ISO Importador','Grupo Importador'], how='left', suffixes=('', '_13'))
df_criterio3['Decile_13'] = df_criterio3['Decile_13'].fillna(0)
df_criterio3 = df_criterio3.query("Decile >= @decil_alto and Decile_13 < @decil_alto")

df_output3 = df_products.merge(df_criterio3[['HS6']].drop_duplicates().reset_index(drop=True), how='inner', on='HS6').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante'], ascending=[False, False])

df_output3.to_excel(carpeta_calidad / 'criterio3.xlsx', index=False)

### Criterio 4: calidad del producto más cercano exportado por Colombia

Para los productos que Colombia aún no exporta, toma como referencia el decil de calidad del
producto más cercano en el espacio de producto que sí se exporta con calidad alta.

In [9]:
df_criterio4 = pd.DataFrame()
for producto in df_products.query("Margen != 'Oportunidades a profundizar'")[['HS6','Margen','Type']].drop_duplicates()['HS6'].tolist():
    df_criterio4 = pd.concat([
            df_criterio4,
            pd.concat(
                [
                    pd.DataFrame([producto], columns=['HS6']),
                    df_calidad.query(f"""`ISO Exportador` == 'COL' and Decile >= {decil_alto} and HS6 == '{nearest_product(df_product_space,producto,list(set(df_calidad.query("`ISO Exportador` == 'COL' and Decile >= 8")['HS6'])),threshold=cercania)[0]}'""").groupby(['HS6'])['Decile'].mean().reset_index().rename(columns={'Decile':'Decile Promedio','HS6':'HS6 Cercano'})
                ]
            ,axis=1)
    ])

df_output4 = df_products.merge(df_criterio4[['HS6','Decile Promedio']].round().drop_duplicates().reset_index(drop=True), how='inner', on='HS6').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante'], ascending=[False, False])

df_output4.to_excel(carpeta_calidad / 'criterio4.xlsx', index=False)

### Criterio 4 en países de comparación

Repite el ejercicio anterior tomando como referencia la calidad con la que Chile, México y Perú
exportan el mismo producto, en lugar del producto más cercano exportado por Colombia.

El notebook original repetía el bloque tres veces, una por país; aquí se define una vez y se
invoca con la tabla de parámetros.

In [10]:
# tabla de parámetros: código ISO del país de comparación y archivo de salida
parametros_comparacion = [
    ('CHL', 'chl_criterio4.xlsx'),
    ('MEX', 'mex_criterio4.xlsx'),
    ('PER', 'per_criterio4.xlsx'),
]

for iso_pais, salida in parametros_comparacion:
    df_criterio4 = pd.DataFrame()
    for producto in df_products.query("Margen != 'Oportunidades a profundizar'")[['HS6','Margen','Type']].drop_duplicates()['HS6'].tolist():
        df_criterio4 = pd.concat([
                df_criterio4,
                pd.concat(
                    [
                        pd.DataFrame([producto], columns=['HS6']),
                        df_calidad.query(f"""`ISO Exportador` == '{iso_pais}' and Decile >= {decil_alto} and HS6 == '{producto}'""").groupby(['HS6'])['Decile'].mean().reset_index().rename(columns={'Decile':'Decile Promedio','HS6':'HS6 Cercano'})
                    ]
                ,axis=1)
        ])

    df_output_pais = df_products.merge(df_criterio4[['HS6','Decile Promedio']].round().drop_duplicates().reset_index(drop=True), how='inner', on='HS6').drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante'], ascending=[False, False])
    df_output_pais.to_excel(carpeta_calidad / 'otros países' / salida, index=False)
    print(f"{salida}: {df_output_pais['HS6'].nunique()} productos")

chl_criterio4.xlsx: 253 productos


mex_criterio4.xlsx: 253 productos


per_criterio4.xlsx: 253 productos
